In [92]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm.auto import tqdm

files = glob.glob(
    "Data/extracted_data_extended/group*/experiment*/subject*.csv"
)

output_dir = "Data/extracted_data_aggregated"

os.makedirs(output_dir, exist_ok=True)

window = pd.Timedelta(milliseconds=500)

for file in tqdm(files, desc="Processing files"):

    print(f"\nProcessing: {file}")
    df = pd.read_csv(file)

    # split image and sensor rows
    images_df = df[df["image_path"].notna()].copy()
    sensors_df = df[df["image_path"].isna()].copy()

    # skip empty
    if len(images_df) == 0:
        continue

    # same time format
    # images_df["time"] = pd.to_datetime(images_df["time"], format="mixed")
    # sensors_df["time"] = pd.to_datetime(sensors_df["time"], format="mixed")

    images_df["time"] = pd.to_timedelta(images_df["time"].astype(str))
    sensors_df["time"] = pd.to_timedelta(sensors_df["time"].astype(str))
    
    images_df = images_df.sort_values("time")
    sensors_df = sensors_df.sort_values("time")

    # SENSOR COLUMNS
    sensor_columns = [
        col for col in sensors_df.columns
        if (
            "value" in col
            and sensors_df[col].notna().any()
        )
    ]

    # AGGREGATION
    aggregated_rows = []

    for _, img_row in tqdm(
        images_df.iterrows(),
        total=len(images_df),
        leave=False,
        desc="Aggregating"
    ):

        t = img_row["time"]

        nearby = sensors_df[
            (sensors_df["time"] >= t - window) &
            (sensors_df["time"] <= t + window)
        ]

        new_row = img_row.to_dict()

        # SENSOR AXIS MEAN / STD

        for col in sensor_columns:

            values = nearby[col].dropna()

            if len(values) == 0:
                new_row[f"{col}_mean"] = np.nan
                new_row[f"{col}_std"] = np.nan

            else:
                new_row[f"{col}_mean"] = values.mean()
                new_row[f"{col}_std"] = values.std()

        # compute magnitude for acceleration and gyroscope
        
        accel_cols = [
            "samsung_linear_acceleration_sensor value0",
            "samsung_linear_acceleration_sensor value1",
            "samsung_linear_acceleration_sensor value2"
        ]

        if all(col in nearby.columns for col in accel_cols):

            accel_valid = nearby[accel_cols].dropna()
            if len(accel_valid) > 0:

                accel_magnitude = np.sqrt(
                    accel_valid[accel_cols[0]]**2 +
                    accel_valid[accel_cols[1]]**2 +
                    accel_valid[accel_cols[2]]**2
                )

                new_row["accel_magnitude_mean"] = accel_magnitude.mean()
                new_row["accel_magnitude_std"] = accel_magnitude.std()

            else:

                new_row["accel_magnitude_mean"] = np.nan
                new_row["accel_magnitude_std"] = np.nan

        gyro_cols = [
            "lsm6dso_gyroscope value0",
            "lsm6dso_gyroscope value1",
            "lsm6dso_gyroscope value2"
        ]

        if all(col in nearby.columns for col in gyro_cols):

            gyro_valid = nearby[gyro_cols].dropna()
            if len(gyro_valid) > 0:

                gyro_magnitude = np.sqrt(
                    gyro_valid[gyro_cols[0]]**2 +
                    gyro_valid[gyro_cols[1]]**2 +
                    gyro_valid[gyro_cols[2]]**2
                )

                new_row["gyro_magnitude_mean"] = gyro_magnitude.mean()
                new_row["gyro_magnitude_std"] = gyro_magnitude.std()

            else:

                new_row["gyro_magnitude_mean"] = np.nan
                new_row["gyro_magnitude_std"] = np.nan

        aggregated_rows.append(new_row)

    
    final_df = pd.DataFrame(aggregated_rows)

    # drop the initial sensor columns (NaNs)
    raw_sensor_cols = [
        col for col in final_df.columns
        if (
            "value" in col
            and not col.endswith("_mean")
            and not col.endswith("_std")
        )
    ]
    
    final_df = final_df.drop(columns=raw_sensor_cols)

    relative_path = os.path.relpath(
        file,
        "Data/extracted_data_extended")

    output_path = os.path.join(
        output_dir,
        relative_path)

    output_path = output_path.replace(
        ".csv",
        ".parquet")

    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True)
    
    final_df.to_parquet(
        output_path,
        engine="pyarrow",
        index=False)

    print(f"[SAVED] {output_path}")
    
    # clean memory
    del df
    del images_df
    del sensors_df
    del final_df

Processing files:   0%|          | 0/2 [00:00<?, ?it/s]


Processing: Data/extracted_data_extended\group01\experiment01\subject_01.csv


Aggregating:   0%|          | 0/2531 [00:00<?, ?it/s]


Processing: Data/extracted_data_extended\group01\experiment01\subject_02.csv


Aggregating:   0%|          | 0/2506 [00:00<?, ?it/s]

In [2]:
import pandas as pd
import numpy as np
import glob

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

files = glob.glob("Data/extracted_data_aggregated/group*/experiment*/subject*.parquet")
df = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
# df.to_parquet("Data/initial_data.parquet", engine="pyarrow", index=False)

## Extract metadata (gender, age, race)

The following cell was executed in Snellius cluster where all the metadata folders for each subject are stored. As an output we receive the same df + 3 columns of age, gender and race for each row. It will output the file "initial_sensors_metadata.parquet" which we use in the subsequent cells.

In [ ]:
# df = pd.read_parquet("Data/initial_data.parquet", engine="pyarrow", index=False)

In [14]:
# Metadata extraction function
def extract_metadata(json_path):
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        face = data.get("person", {}).get("face", {})

        age = face.get("age", None)
        gender = face.get("gender", {}).get("gender_name", None)
        race = face.get("race", {}).get("dominant_race", None)

        # race probabilities
        prob_race = face.get("race", {}).get("probability_race", {})

        return (
            age,
            gender,
            race,
            prob_race.get("asian", None),
            prob_race.get("indian", None),
            prob_race.get("black", None),
            prob_race.get("white", None),
            prob_race.get("middle eastern", None),
            prob_race.get("latino hispanic", None),
        )

    except Exception:
        return (None,) * 9

# Parallel processing
def process_dataframe(df, n_jobs):
    paths = df["metadata"].tolist()
    results = []

    for i, p in enumerate(paths):
        results.append(extract_metadata(p))

        if i % 5000 == 0:
            print(f"Processed {i}/{len(paths)} rows")

    df[[
    "age",
    "gender_name",
    "race",
    "race_asian",
    "race_indian",
    "race_black",
    "race_white",
    "race_middle_eastern",
    "race_latino_hispanic"
    ]] = pd.DataFrame(results, index=df.index)
    return df

def main():
    start_time = time.time()
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True, help="Input dataframe path (parquet)")
    parser.add_argument("--output", required=True, help="Output dataframe path")
    parser.add_argument("--n_jobs", type=int, default=16)

    args = parser.parse_args()

    print("Loading dataframe...")
    df = pd.read_parquet(args.input)

    print(f"Processing {len(df)} rows with {args.n_jobs} workers...")

    df = process_dataframe(df, args.n_jobs)

    print("Saving output...")
    df.to_parquet(args.output)

    print("Done.")
    print(f"Elapsed: {time.time() - start_time:.2f}s")

# df.to_parquet("Data/initial_metadata.parquet", engine="pyarrow")

### Drop records with noise in image

In [83]:
df = pd.read_parquet("Data/initial_metadata.parquet", engine="pyarrow")

In [84]:
df.isna().sum()

group                        0
time                         0
image_path                   0
metadata                     0
subject                      0
                         ...  
race_black              146879
race_indian             146879
race_latino_hispanic    146879
race_middle_eastern     146879
race_white              146879
Length: 69, dtype: int64

In [ ]:
import tarfile
from tqdm import tqdm
import os

The objective of this process is to identify frames that could introduce noise in the pipeline, such as images where the face is non visible or covered. 

The following cells were executed in Snellius cluster where all the metadata folders for each subject are stored in tar fornat. As an output we receive the same df + 2 columns of valid (True - False), and the reason if False. It will output the file "initial_sensors_metadata_valid.parquet" which we use in the subsequent cells.

metadata folders are in format "metadata.tar" for each subject due to inode constraints for Snellius.

In [ ]:
def get_tar_and_filename(metadata_path):
    """
    Input:
        /.../subject_01/metadata/10_59_49_000776.json

    Output:
        (/.../subject_01/metadata.tar, 10_59_49_000776.json)
    """

    parts = metadata_path.split("/")

    try:
        meta_idx = parts.index("metadata")
    except ValueError:
        return None, None

    base = "/".join(parts[:meta_idx])  # up to subject_01
    filename = parts[-1]

    tar_path = f"{base}/metadata.tar"

    return tar_path, filename

def load_tar_index(tar_path):
    tar = tarfile.open(tar_path, "r")
    index = {}

    for m in tar.getmembers():
        if m.isfile() and m.name.endswith(".json"):
            fname = m.name.split("/")[-1]
            index[fname] = m

    return tar, index


def read_metadata_from_tar(tar, member):
    f = tar.extractfile(member)
    if f is None:
        return None
    return json.load(f)

In [ ]:
def is_valid_frame(metadata_path, metadata):

    if metadata is None:
        print(metadata_path, "rejected: metadata is None")
        return False, "metadata_none"

    person = metadata.get("person")
    # if person is None:
    #     print(metadata_path, "rejected: missing person")
    #     return False, "missing_person"

    face = person.get("face")
    if face is None:
        print(metadata_path, "rejected: missing face")
        return False, "missing_face"

    # --- Face bounding box ---
    bbox = face.get("bounding_box")
    
    if bbox is None:
        print(metadata_path, "rejected: missing bbox")
        return False, "missing_bbox"
    
    # CASE 1: dict format
    if isinstance(bbox, dict):
    
        x0 = bbox.get("x0")
        y0 = bbox.get("y0")
        x1 = bbox.get("x1")
        y1 = bbox.get("y1")
    
    # CASE 2: list format
    elif isinstance(bbox, list):
    
        if len(bbox) != 4:
            print(metadata_path, "rejected: malformed bbox list")
            return False, "malformed_bbox"
    
        x0, y0, x1, y1 = bbox
    
    # UNKNOWN FORMAT
    else:
        print(metadata_path, "rejected: unknown bbox format")
        return False, "unknown_bbox_format"
    
    # validate coords
    if None in (x0, y0, x1, y1):
        print(metadata_path, "rejected: incomplete bbox")
        return False, "incomplete_bbox"
    
    width = x1 - x0
    height = y1 - y0
    
    if width <= 0 or height <= 0:
        print(metadata_path, "rejected: invalid bbox dimensions")
        return False, "invalid_bbox"

    return True, "valid"

In [ ]:
results = []
reasons = []

tar_cache = {}  # avoid reopening same tar

for _, row in tqdm(df.iterrows(), total=len(df)):

    metadata_path = row["metadata"]

    # Resolve tar path + filename
    tar_path, filename = get_tar_and_filename(metadata_path)

    if tar_path is None:
        results.append(None)
        reasons.append("invalid_tar_path")
        continue

    # Open tar (cached)
    try:

        if tar_path not in tar_cache:
            tar, index = load_tar_index(tar_path)
            tar_cache[tar_path] = (tar, index)

        tar, index = tar_cache[tar_path]

    except Exception as e:

        print(f"TAR OPEN ERROR: {metadata_path} -> {e}")

        results.append(None)
        reasons.append("tar_open_failure")
        continue

    # Find json inside tar
    member = index.get(filename)

    if member is None:

        print(f"MISSING JSON IN TAR: {metadata_path}")

        results.append(None)
        reasons.append("missing_json_in_tar")
        continue

    # Read metadata
    try:

        metadata = read_metadata_from_tar(tar, member)

    except Exception as e:

        print(f"METADATA READ ERROR: {metadata_path} -> {e}")

        results.append(None)
        reasons.append("metadata_read_failure")
        continue

    # Empty metadata
    if metadata is None or metadata == {}:

        print(f"EMPTY METADATA: {metadata_path}")

        results.append(None)
        reasons.append("empty_metadata")
        continue

    # Actual frame validation
    try:

        valid, reason = is_valid_frame(metadata_path, metadata)

        results.append(valid)
        reasons.append(reason)

    except Exception as e:

        print(f"VALIDATION ERROR: {metadata_path} -> {e}")

        results.append(None)
        reasons.append("validation_failure")


for tar, _ in tar_cache.values():
    tar.close()


df["valid"] = results
df["invalid_reason"] = reasons

In [ ]:
# df.to_parquet("Data/dipser_data_valid.parquet", engine="pyarrow", index=False)

### Data Cleaning

In [3]:
df = pd.read_parquet("Data/dipser_data_valid.parquet", engine="pyarrow")

In [4]:
df['invalid_reason'].value_counts()

invalid_reason
valid                    1059808
missing_face               46881
missing_bbox                2714
metadata_read_failure       2082
validation_failure             8
Name: count, dtype: int64

In [5]:
# the "None" values where with cases where metadata was empty. we set these rows to True as there is no evidence they are noise
# cases where invalid_reason in ['metadata_read_failure', 'validation_failure']
df["valid"] = df["valid"].astype("boolean").fillna(True)

In [6]:
df.shape

(1111493, 71)

In [7]:
# remove rows with noisy images
df = df[df['valid'] == True]

In [8]:
df.shape

(1061898, 71)

### Transform Metadata

After removing noisy images, with the remaining rows we can compute the fairness labels of gender, age and race

In [9]:
race_cols = [
    "race_asian",
    "race_indian",
    "race_black",
    "race_white",
    "race_middle_eastern",
    "race_latino_hispanic"
]

In [10]:
subject_probs = (
    df.groupby(["group", "subject"])[race_cols]
    .mean()
    .reset_index()
)

In [11]:
"""Deepface extracts per each timeframe the estimation for race, gender and age.
 For the column of gender, we will keep the most common label for the whole subject."""

# we will groupby group experiment and subject and we will keep the most dominant 
def get_mode(series):
    return series.dropna().mode().iloc[0] if not series.dropna().empty else None

In [12]:
def get_dominant_race(row):
    return row[race_cols].idxmax().replace("race_", "")

In [13]:
subject_probs["race"] = subject_probs.apply(get_dominant_race, axis=1)
subject_probs = subject_probs.drop(columns=race_cols)

In [14]:
subject_metadata = (
    df.groupby(["group", "subject"])
    .agg({
        "gender_name": get_mode,
        "age": "mean"
    })
    .reset_index())

# merge with race
subject_metadata = subject_metadata.merge(
    subject_probs,
    on=["group", "subject"],
    how="left"
)

In [15]:
df = df.drop(columns=['race', 'gender_name', 'age']) # will be replaced with the new values
df = df.merge(
    subject_metadata,
    on=["group", "subject"],
    how="left")

In [16]:
df['age'] = round(df['age'])

In [17]:
df = df.drop(columns=race_cols)

### Remove subjects with missing sensors

In [18]:
df['samsung_hr_none_wakeup_sensor value0_mean'].isna().sum()

np.int64(155464)

In [19]:
df['subject_experiment_id'] = df['group'] + "_" + df['experiment'] + '_' + df['subject'] 

In [20]:
missing_ratio = (
    df.groupby('subject_experiment_id')['samsung_hr_none_wakeup_sensor value0_mean']
    .apply(lambda x: x.isna().mean())
)
missing_ratio.sort_values(ascending=False).head()

subject_experiment_id
group02_experiment08_subject_01    1.0
group02_experiment08_subject_10    1.0
group02_experiment09_subject_01    1.0
group01_experiment05_subject_14    1.0
group01_experiment05_subject_01    1.0
Name: samsung_hr_none_wakeup_sensor value0_mean, dtype: float64

In [21]:
len(set(df['subject_experiment_id']))

483

In [22]:
MISSING_SENSORS_THRESHOLD = 0.3

In [23]:
valid_subjects = missing_ratio[missing_ratio < MISSING_SENSORS_THRESHOLD].index
df = df[df["subject_experiment_id"].isin(valid_subjects)]

In [24]:
len(set(df['subject_experiment_id']))

408

In [25]:
df['samsung_hr_none_wakeup_sensor value0_mean'].isna().sum()

np.int64(10310)

In [28]:
df.shape

(899997, 66)

In [26]:
df = df.dropna(subset='samsung_hr_none_wakeup_sensor value0_mean')
df = df.dropna(subset='samsung_rotation_vector value0_mean') # few records

In [28]:
df = df.drop(columns='valid')

### Set "Ground Truth" Attention label

In [29]:
attention_cols = [
    col for col in df.columns
    if 'attentionfilled' in col and 'self' not in col]

# use averaging to serve as a ground truth label
df['attention'] = df[attention_cols].mean(axis=1)

In [30]:
df.columns

Index(['group', 'time', 'image_path', 'metadata', 'subject', 'experiment',
       'self_labeling emotion', 'self_labeling attention',
       'labeler_02 attention', 'labeler_04 attention', 'labeler_02 emotion',
       'labeler_01 attention', 'labeler_03 emotion', 'labeler_03 attention',
       'labeler_01 emotion', 'labeler_04 emotion',
       'self_labeling emotionfilled', 'self_labeling attentionfilled',
       'labeler_02 attentionfilled', 'labeler_04 attentionfilled',
       'labeler_02 emotionfilled', 'labeler_01 attentionfilled',
       'labeler_03 emotionfilled', 'labeler_03 attentionfilled',
       'labeler_01 emotionfilled', 'labeler_04 emotionfilled',
       'samsung_rotation_vector value0_mean',
       'samsung_rotation_vector value0_std',
       'samsung_rotation_vector value1_mean',
       'samsung_rotation_vector value1_std',
       'samsung_rotation_vector value2_mean',
       'samsung_rotation_vector value2_std',
       'samsung_rotation_vector value3_mean',
       'sam

In [34]:
# emotion based on consensus
df['attention_target'] = df[['labeler_01 attentionfilled', 'labeler_02 attentionfilled',  'labeler_03 attentionfilled', 'labeler_04 attentionfilled',  'labeler_05 attentionfilled']].mode(axis=1)[0]

### Unique identifier for each subject

In [31]:
df['subject_id'] = df['group'] + "_" + df['subject'] 

In [32]:
df = df.rename(columns={'samsung_hr_none_wakeup_sensor value0_mean': 'heart_rate', 'samsung_hr_none_wakeup_sensor value0_std': 'heart_rate_std', 'gender_name': 'gender'})

In [33]:
# remaining nans only for cases where one of the labellers was not part of the evaluation of the subject
df.isna().sum().head(60)

group                                                  0
time                                                   0
image_path                                             0
metadata                                               0
subject                                                0
experiment                                             0
self_labeling emotion                             895824
self_labeling attention                           896161
labeler_02 attention                              897674
labeler_04 attention                              898354
labeler_02 emotion                                893692
labeler_01 attention                              881632
labeler_03 emotion                                891690
labeler_03 attention                              891053
labeler_01 emotion                                896225
labeler_04 emotion                                898339
self_labeling emotionfilled                         2018
self_labeling attentionfilled  

### Add extra timestamp in group/experiment/subject level

In [34]:
df["time_sec"] = (pd.to_datetime(df["time"]))

# convert it to 0-300
df["time_sec"] = (
    df.groupby("subject_experiment_id")["time_sec"]
    .transform(lambda x:
        (x - x.min())
        .dt.total_seconds()
        .astype(int)))


# move 'time_sec' to 3rd column position
cols = df.columns.tolist()
cols.remove("time_sec")
cols.insert(2, "time_sec")
df.head()

,group,time,image_path,metadata,subject,experiment,self_labeling emotion,self_labeling attention,labeler_02 attention,labeler_04 attention,labeler_02 emotion,labeler_01 attention,labeler_03 emotion,labeler_03 attention,labeler_01 emotion,labeler_04 emotion,self_labeling emotionfilled,self_labeling attentionfilled,labeler_02 attentionfilled,labeler_04 attentionfilled,labeler_02 emotionfilled,labeler_01 attentionfilled,labeler_03 emotionfilled,labeler_03 attentionfilled,labeler_01 emotionfilled,labeler_04 emotionfilled,samsung_rotation_vector value0_mean,samsung_rotation_vector value0_std,samsung_rotation_vector value1_mean,samsung_rotation_vector value1_std,samsung_rotation_vector value2_mean,samsung_rotation_vector value2_std,samsung_rotation_vector value3_mean,samsung_rotation_vector value3_std,samsung_rotation_vector value4_mean,samsung_rotation_vector value4_std,lsm6dso_gyroscope value0_mean,lsm6dso_gyroscope value0_std,lsm6dso_gyroscope value1_mean,lsm6dso_gyroscope value1_std,lsm6dso_gyroscope value2_mean,lsm6dso_gyroscope value2_std,samsung_linear_acceleration_sensor value0_mean,samsung_linear_acceleration_sensor value0_std,samsung_linear_acceleration_sensor value1_mean,samsung_linear_acceleration_sensor value1_std,samsung_linear_acceleration_sensor value2_mean,samsung_linear_acceleration_sensor value2_std,opt3007_light value0_mean,opt3007_light value0_std,heart_rate,heart_rate_std,accel_magnitude_mean,accel_magnitude_std,gyro_magnitude_mean,gyro_magnitude_std,labeler_05 emotion,labeler_05 attention,labeler_05 emotionfilled,labeler_05 attentionfilled,invalid_reason,gender,age,race,subject_experiment_id,attention,subject_id,time_sec
0,group01,2026-05-08 10:40:43.047895,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_047895.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_047895.json,subject_01,experiment01,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109524,0.000163,-0.659717,0.000588,0.632875,0.000495,0.390186,0.000218,246.0,0.0,-0.006022,0.006482,-0.003838,0.003665,-0.000864,0.006845,0.052984,0.034713,-0.132399,0.047077,0.017549,0.043508,63.0,0.0,78.0,NaN,0.155556,0.041540,0.011095,0.005498,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,0
1,group01,2026-05-08 10:40:43.195317,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_195317.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_195317.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109485,0.000109,-0.660015,0.000520,0.632620,0.000426,0.390106,0.000225,246.0,0.0,-0.004508,0.005235,-0.004068,0.003267,-0.001026,0.005944,0.053056,0.036979,-0.124020,0.045047,0.012282,0.043478,63.0,0.0,78.0,NaN,0.148634,0.038779,0.009572,0.004378,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,0
2,group01,2026-05-08 10:40:43.295405,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_295405.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_295405.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109478,0.000101,-0.660194,0.000514,0.632473,0.000406,0.390043,0.000252,246.0,0.0,-0.004581,0.005193,-0.003897,0.003153,-0.000525,0.005644,0.054420,0.036670,-0.129478,0.044805,0.015730,0.040968,63.0,0.0,78.0,NaN,0.153129,0.038995,0.009384,0.004079,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,0
3,group01,2026-05-08 10:40:43.395835,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_395835.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_395835.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [ ]:
# unusual values for that experiment subject
df = df[df['subject_experiment_id'] != 'group01_experiment04_subject_15']

### Export to Parquet

In [35]:
df.to_parquet("Data/dipser_dataset.parquet", engine="pyarrow", index=False)